In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import openai
import os

if not os.path.exists("test_notebooks"):
    os.chdir("..")

assert os.path.exists("test_notebooks")

In [ ]:

client = openai.OpenAI(
    # This is the default and can be omitted
    base_url = "http://localhost:8080/api",
    api_key= "sk-9f605d210924406d8e60b10122f59e96", # open-webui, safe
)

In [ ]:
autodl_kimi = "Kimi-K2.5" # 这个是autodl的

completion = client.chat.completions.create(
    model=autodl_kimi,
    messages=[
        {
            "role": "user",
            "content": "1+1=?",
        },
    ],
)

print(completion.choices[0].message.content)

In [ ]:
# demo

import base64
import os
from pathlib import Path
import io
from PIL import Image


def encode_imgfile_to_url(file: str, quality=90, debug=False) -> str:
    with io.BytesIO() as buf:
        with Image.open(file) as im:
            im.convert("RGB").save(buf, format="JPEG", quality=quality)

            if debug:
                Path("jpg-debug.tmp.jpg").write_bytes(buf.getvalue())

            image_url = f"data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode('utf-8')}"

            return image_url

a = encode_imgfile_to_url("runs/episodes/miyako_1/images/t008_02.png", debug=True)
a.__len__() // 1024, "KB-b64"

In [ ]:
completion = client.chat.completions.create(
    model=autodl_kimi,
    temperature=0.6,
    extra_body={
        "thinking": {"type": "disabled"},
    },
    messages=[
        {
            "role": "user",
            # 注意这里，content 由原来的 str 类型变更为一个 list，这个 list 中包含多个部分的内容，图片（image_url）是一个部分（part），
            # 文字（text）是一个部分（part）
            "content": [
                {
                    "type": "text",
                    "text": "img_g1:",
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": encode_imgfile_to_url(r"runs\episodes\char_sora\t003_01.png"),
                    },
                },
                {
                    "type": "text",
                    "text": "img_x1:",
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": encode_imgfile_to_url("runs/episodes/miyako_1/images/t008_02.png"),
                    },
                },
                {
                    "type": "text",
                    "text": "你看了哪几个img？名称叫什么？简单描述一下区别？"
                }
            ],
        },
    ],
)
 
print(completion.choices[0].message.content)
print("usage", completion.usage)

artist only phase comparison

In [ ]:
from typing import List


test_case = {
    "ref_image": r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t004_02.png",
    "ref_text": "我喜欢这种美少女图片",
    "candidates": [
        r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t001_03.png",  # 还行 [0]
        r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t002_00.png",  # 不好 [1]
        r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t002_01.png",  # 不好 [2]
        r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t002_03.png",  # 还行 [3]
    ],
}


def llm_compare_image_artist_only(ref_image: str, ref_text: str, candidates: List[str]) -> str:
    system_message = """
## 角色
你是一个二次元图像审美评判专家。你需要为用户选择更好的图片。用户会提交用户需求。
用户需求由 参考图片ref_image、文本需求组成。

除此之外，你还有 待评价图片 candidate[i] 作为输入。

你要做的是，根据用户的参考图片、文本需求，将待评价图片进行排序，由好到差。

## 输出
1. 分析并说明理由
2. 得出结论：尝试推测用户喜好。待评价图片从好到差。你需要寻找用户可能最喜欢的图片。以代码块compare的格式输出。

```compare
candidate[?] > candidate[?] > ...
```

"""
    user_image_contents = [
        {
            "type": "text",
            "text": f"用户需求文本: {ref_text}, 用户提供的参考图片 ref_image (不参与评分排序):",
        },
        {
            "type": "image_url",
            "image_url": {
                "url": encode_imgfile_to_url(ref_image),
            },
        },
        {
            "type": "text",
            "text": "\n------\n以下是所有待评价图片(candidate):"
        }
    ]

    for i, candidate in enumerate(candidates):
        user_image_contents += [
            {
                "type": "text",
                "text": f"candidate[{i}]:",
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": encode_imgfile_to_url(candidate),
                },
            },
        ]

    completion = client.chat.completions.create(
        model=autodl_kimi,
        temperature=0.6,
        extra_body={
            "thinking": {"type": "disabled"},
        },
        messages=[
            {
                "role": "system",  # type: ignore
                "content": system_message,
            },
            {
                "role": "user",
                "content": user_image_contents,
            },
        ],
    )

    print("usage:", completion.usage)

    s = completion.choices[0].message.content
    assert s
    return s

out = llm_compare_image_artist_only(test_case["ref_image"], test_case["ref_text"], test_case["candidates"] )
print(out)